In [ ]:
!pip install autogen
!pip install --upgrade pydantic
!pip install ipywidgets
!pip install groq

In [ ]:
from typing import Literal
import autogen
from typing_extensions import Annotated
import getpass

In [ ]:
# Solicitar a API key de forma segura
from google.colab import userdata
api_key =  userdata.get('groq')

config_list = [
    {
      "model": "openai/gpt-oss-20b",  # ""llama-3.3-70b-versatile",  # "llama3-8b-8192",
      "api_key": api_key,
      "api_type": "groq"
    }
]

In [ ]:
llm_config = {
    "config_list": config_list,
    "timeout": 120
}

#agente que efetua conversão de moeda
currency_bot = autogen.AssistantAgent(
    name="currency_bot",
    system_message="Para tarefas relacionadas com conversão de moedas, use apenas as funções que você recebeu. Responda TERMINATE "
                   "quando a tarefa estiver concluída.",
    llm_config=llm_config
)

#agente user proxy
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="NEVER",
    max_consecutive_auto_reply=2,
    code_execution_config=False
)

CurrencySymbol = Literal["USD", "EUR"]

#função de taxa_de_câmbio
def exchange_rate(base_currency: CurrencySymbol, quote_currency: CurrencySymbol) -> float:
    if base_currency == quote_currency:
        return 1.0
    
    elif base_currency == "USD" and quote_currency == "EUR":
        return 1 / 1.09
    
    elif base_currency == "EUR" and quote_currency == "USD":
        return 1 / 1.1
    
    else:
        raise ValueError(f"Moedas desconhecidas: {base_currency}, {quote_currency}")  # Tradução: "Moedas desconhecidas:..."

#registrando a ferramenta
@user_proxy.register_for_execution()
# Register the tool signature with the assistant agent.
@currency_bot.register_for_llm(description="Calculadora de câmbio de moeda")  # Tradução: "Calculadora de câmbio de moeda"
def currency_calculator(
        base_amount: Annotated[float, "Quantia da moeda em base_currency"],  # Tradução: "Quantia da moeda em base_currency"
        base_currency: Annotated[CurrencySymbol, "Moeda base"] = "USD",  # Tradução: "Moeda base"
        quote_currency: Annotated[CurrencySymbol, "Moeda de cotação"] = "EUR"  # Tradução: "Moeda de cotação"
) -> str:
    quote_amount = exchange_rate(base_currency, quote_currency) * base_amount
    return f"{quote_amount} - {quote_currency}"



#######################################
#inicialização da conversa
chat = user_proxy.initiate_chat(
    currency_bot,
    message="E ai, pai? Tenho 200 dolares e estou indo pra europa, quanto eu tenho na moeda local?"  # Tradução: "Converter 100 USD para EUR"
    # message="E ai, truta! Está calor, hein? Estou na sala"

)


In [ ]:
llm_config = {
    "config_list": config_list,
    "timeout": 120
}

#agente que efetua conversão de moeda
currency_bot = autogen.AssistantAgent(
    name="currency_bot",
    system_message=("Atue como um assitente gentil de automação residencial com acesso a funções."
                    "Use apenas as funções relacionadas com automação que você recebeu. "
                    "Adiante os desejos do usuário e responda de forma gentil. "
                    "Responda TERMINATE quando a tarefa estiver concluída."),
    llm_config=llm_config
)

#agente user proxy
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=1,
    code_execution_config=False
)

EnviromentSymbol = Literal["Sala de estar", "quarto", "cozinha"]

#registrando a ferramenta
@user_proxy.register_for_execution()
# Register the tool signature with the assistant agent.
@currency_bot.register_for_llm(description="Liga ou desliga o ar condicionado de algum ambiente")
def turn_on_airconditioner(
        ambiente: Annotated[EnviromentSymbol, "Ambiente que é necessário ligar ou desligar o ar condicionado"],
        estado: Annotated[bool, "True para ligar o ar condicionado, False para desligar"]
) -> str:
    return f"Ar condicionado do {ambiente} foi {'ligado' if estado else 'desligado'}"


#######################################
#inicialização da conversa
chat = user_proxy.initiate_chat(
    currency_bot,
    # message="E ai, pai? Tenho 200 dolares e estou indo pra europa, quanto eu tenho na moeda local?"  # Tradução: "Converter 100 USD para EUR"
    message="E ai, truta! Está calor, hein? Estou na sala"

)